# Object Detection

## Setup

In [ ]:
!uv pip install python-dotenv ultralytics torch fiftyone -qq

In [ ]:
import os
import yaml
import torch
from dotenv import load_dotenv

import fiftyone as fo
import fiftyone.zoo as foz
from ultralytics import YOLO

load_dotenv()

EXPORT_DIR = os.getenv("EXPORT_DIR")
yaml_path = os.path.join(EXPORT_DIR, "dataset.yaml")

wandb_project_name = 'coco-yolov8'
os.environ['WANDB_PROJECT'] = wandb_project_name
os.environ['WANDB_MODE'] = 'offline'

CLASSES = ["car", "motorcycle", "bus", "truck"]

In [ ]:
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Data
*only needs to be run once per machine*

In [ ]:
# Load the needed COCO-2017 splits
train_dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="train",
    dataset_name="coco-2017-train",
    # max_samples=1000,  # uncomment to limit for faster iteration
    classes=CLASSES,
)

val_dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="validation",
    dataset_name="coco-2017-val",
    # max_samples=500,
    classes=CLASSES,
)

In [ ]:
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")

In [ ]:
# FiftyOne can export directly to YOLOv5-format (works with YOLOv8)
train_dataset.export(
    export_dir=EXPORT_DIR,
    dataset_type=fo.types.YOLOv5Dataset,
    split="train",
    label_field="ground_truth",   # COCO zoo dataset uses this field
    classes=CLASSES
)

val_dataset.export(
    export_dir=EXPORT_DIR,
    dataset_type=fo.types.YOLOv5Dataset,
    split="val",
    label_field="ground_truth",
    classes=CLASSES,
)

In [ ]:
dataset_yaml = {
    "path": EXPORT_DIR,
    "train": "images/train",
    "val": "images/val",
    "nc": len(CLASSES),
    "names": CLASSES,
}

with open(yaml_path, "w") as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print(f"Number of classes: {len(CLASSES)}")

## Training
### yolov8m

In [ ]:
run_name = "run-yolov8m-full"

In [ ]:
model = YOLO("yolov8m.pt")

results = model.train(
    data=yaml_path,
    epochs=100,           # increase for better results
    imgsz=640,
    batch=96,            # let ultralytics decide optimal
    device=0,            # 0 = first GPU
    project=wandb_project_name,
    name=run_name,
    save=True,
    save_period=10,          # checkpoint every 10 epochs
    patience=10,         # early stopping
    workers=16,           # Colab has limited CPU cores

    optimizer="AdamW",
    lr0=3.5e-3,
    lrf=0.05,                # final lr = lr0 * lrf
    cos_lr=True,
    warmup_epochs=3,
    weight_decay=5e-4,
    amp=True,                # automatic mixed precision (BF16 on A100) — free speedup

    close_mosaic=10,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,          # good for occluded vehicle scenarios

    # Augmentation (helps generalize to varied vehicle appearances)
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.4,
)

In [ ]:
val_dataset = fo.load_dataset("coco-2017-val")
best_ckpt = os.path.join(EXPORT_DIR, "runs", "detect", wandb_project_name, run_name, "weights", "best.pt")
trained_model = YOLO(best_ckpt)

metrics = trained_model.val(data=yaml_path)
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

In [ ]:
import fiftyone as fo

# Run inference on the val set
val_dataset = fo.load_dataset("coco-2017-val")
trained_model = YOLO(best_ckpt)

for sample in val_dataset.iter_samples(autosave=True):
    result = trained_model(sample.filepath, verbose=False)[0]
    detections = []
    h, w = result.orig_shape
    for box, conf, cls in zip(
        result.boxes.xyxy.cpu().numpy(),
        result.boxes.conf.cpu().numpy(),
        result.boxes.cls.cpu().numpy(),
    ):
        x1, y1, x2, y2 = box
        detections.append(
            fo.Detection(
                label=result.names[int(cls)],
                bounding_box=[x1 / w, y1 / h, (x2 - x1) / w, (y2 - y1) / h],
                confidence=float(conf),
            )
        )
    sample["yolov8_preds"] = fo.Detections(detections=detections)

# Launch the FiftyOne app (works inline in Colab)
session = fo.launch_app(val_dataset, auto=False)

In [ ]:
from fiftyone.utils.annotations import draw_labeled_images

VIZ_OUTPUT = os.path.join(EXPORT_DIR, "viz_output")

draw_labeled_images(
    val_dataset.take(50),
    VIZ_OUTPUT,
    label_fields=["ground_truth", "yolov8_preds"],
)

### yolov8s

In [ ]:
run_name = "run-yolov8s-full"

In [ ]:
# Since we are only training to detect four classes, we will see if
# a smaller model can match the performance of the bigger one.
model = YOLO("yolov8s.pt")

# These are the exact same training arguments for fair comparison
results = model.train(
    data=yaml_path,
    epochs=100,           # increase for better results
    imgsz=640,
    batch=96,            # let ultralytics decide optimal
    device=0,            # 0 = first GPU
    project=wandb_project_name,
    name=run_name,
    save=True,
    save_period=10,          # checkpoint every 10 epochs
    patience=10,         # early stopping
    workers=16,           # Colab has limited CPU cores

    optimizer="AdamW",
    lr0=3.5e-3,
    lrf=0.05,                # final lr = lr0 * lrf
    cos_lr=True,
    warmup_epochs=3,
    weight_decay=5e-4,
    amp=True,                # automatic mixed precision (BF16 on A100) — free speedup

    close_mosaic=10,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,          # good for occluded vehicle scenarios

    # Augmentation (helps generalize to varied vehicle appearances)
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.4,
)

In [ ]:
yaml_path = os.path.join(EXPORT_DIR, "dataset.yaml")
val_dataset = fo.load_dataset("coco-2017-val")
best_ckpt = os.path.join(EXPORT_DIR, "runs", "detect", wandb_project_name, run_name, "weights", "best.pt")
trained_model = YOLO(best_ckpt)

metrics = trained_model.val(data=yaml_path)
print("yolov8s")
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

In [ ]:
run_name = "run-yolov8s-full2"

In [ ]:
model = YOLO("yolov8s.pt")

# Changing some training arguments more specific to the smaller model
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=96,  # bigger batch size
    device=0,
    project=wandb_project_name,
    name=run_name,
    save=True,
    save_period=10,
    patience=10,
    workers=16,

    optimizer="AdamW",
    lr0=1.0e-3,  # learning rate used in yolo pretraining
    lrf=0.05,
    cos_lr=True,
    warmup_epochs=3,
    weight_decay=5e-4,
    amp=True,

    close_mosaic=10,
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.05,

    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.4,
)

In [ ]:
yaml_path = os.path.join(EXPORT_DIR, "dataset.yaml")
val_dataset = fo.load_dataset("coco-2017-val")
best_ckpt = os.path.join(EXPORT_DIR, "runs", "detect", wandb_project_name, run_name, "weights", "best.pt")
trained_model = YOLO(best_ckpt)

metrics = trained_model.val(data=yaml_path)
print("yolov8s_2")
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

The different training arguments don't appear to make much of a difference, indicating the difference in performance is largely attributable to model size. Since the smaller model has less than half as many parameters as the larger (i.e., m) model, we will stick with the smaller one to give us more memory headroom when it is put on device.